### In this module:
In this notebook we will show the basics of using Double-ML tools through the `econml` Python package (other popular package is [causalml](https://github.com/uber/causalml/)). We will
-  Estimate an Average Treatment Effect (ATE) controlling for confounding.
- Estimate a Conditional Average Treatment Effect (CATE) controlling for confounding.

This will be short notebook (with no exercise) as the use-cases are very similar to Regression/Prediction from Exercise #2.

In [ ]:
# Install required packages if needed (pinning latest when tested)
!pip install EconML==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: shap
    Found existing installation: shap 0.51.0
    Uninstalling shap-0.51.0:
      Successfully uninstalled shap-0.51.0


In [2]:
import random
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor

random.seed(2718281)
np.random.seed(2718281)

For data we will use the classic Dominick's OJ dataset.

In [6]:
import os
import urllib.request
if not os.path.exists('oj_large.csv'):
    urllib.request.urlretrieve("https://github.com/bquistorff/NABE_workbooks/raw/colab/oj_large.csv", 'oj_large.csv')

In [7]:
oj_data = pd.read_csv("oj_large.csv")

In [8]:
oj_data.head()

,store,brand,week,logmove,feat,price,AGE60,EDUC,ETHNIC,INCOME,HHLARGE,WORKWOM,HVAL150,SSTRDIST,SSTRVOL,CPDIST5,CPWVOL5
0,2,tropicana,40,9.018695,0,3.87,0.232865,0.248935,0.11428,10.553205,0.103953,0.303585,0.463887,2.110122,1.142857,1.92728,0.376927
1,2,tropicana,46,8.723231,0,3.87,0.232865,0.248935,0.11428,10.553205,0.103953,0.303585,0.463887,2.110122,1.142857,1.92728,0.376927
2,2,tropicana,47,8.253228,0,3.87,0.232865,0.248935,0.11428,10.553205,0.103953,0.303585,0.463887,2.110122,1.142857,1.92728,0.376927
3,2,tropicana,48,8.987197,0,3.87,0.232865,0.248935,0.11428,10.553205,0.103953,0.303585,0.463887,2.110122,1.142857,1.92728,0.376927
4,2,tropicana,50,9.093357,0,3.87,0.232865,0.248935,0.11428,10.553205,0.103953,0.303585,0.463887,2.110122,1.142857,1.92728,0.376927


We will focus on how log price might impact the log of quantity sold, taking into account of variables in the dataset.

In [9]:
Y = oj_data['logmove'].values
T = np.log(oj_data["price"]).values
scaler = StandardScaler()
W1 = scaler.fit_transform(oj_data[[c for c in oj_data.columns
                                   if c not in ['price', 'logmove', 'brand', 'week', 'store','INCOME']]].values)
W2 = pd.get_dummies(oj_data[['brand']]).values # turn brand into dummies
W = np.concatenate([W1, W2], axis=1)
X=scaler.fit_transform(oj_data[['INCOME']].values)

#### Constant Average Treatment Effect
At first we will assume a constant Average Treatment Effect (ATE) controlling for confounders $W$. We will use a random forest for both the outcome and treatment models.

In [ ]:
est1 = LinearDML(model_y=RandomForestRegressor(),model_t=RandomForestRegressor())
ef1 = est1.fit(Y, T, W=W)
print(f"ATE={ef1.ate()} and ATE CI={ef1.ate_interval(alpha=0.05)}")

ATE=-2.661016593501461 and ATE CI=(np.float64(-2.704604604890571), np.float64(-2.6174285821123506))


#### Conditional Average Treatment Effect
We can also look at how you can estimate a treatment effect that may vary based on additional variables $X$. It will return an estimated treatment effect for each $X_i$.

In [24]:
est2 = LinearDML(model_y=RandomForestRegressor(),model_t=RandomForestRegressor())
ef2 = est2.fit(Y, T, W=W, X=X)
ef_Xs =  ef2.effect(X)
print(X.shape, ef_Xs.shape)

(28947, 1) (28947,)


There are many more model options! For example, [DRIV](https://www.pywhy.org/EconML/_autosummary/econml.iv.dr.LinearDRIV.html) for Instrumental Variables. See a demo [here](https://github.com/py-why/EconML/blob/main/notebooks/Double%20Machine%20Learning%20Examples.ipynb) and a flowchart of models to use [here](https://www.pywhy.org/EconML/spec/flowchart.html).

In [14]:
# Reproducibility info
import sys, sklearn, econml
print(f'Python:       {sys.version}')
print(f'NumPy:        {np.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'econml: {econml.__version__}')

Python:       3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
NumPy:        2.0.2
scikit-learn: 1.6.1
econml: 0.16.0
